In [1]:
# Importar las librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
# URL del archivo en el bucket público de AWS
url = "https://hybridge-education-machine-learning-datasets.s3.us-east-1.amazonaws.com/Fraud.csv"
df = pd.read_csv(url)

In [3]:
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [4]:
len(df)

6362620

# ¡¡¡6,362,620 OBSERVACIONES!!!

In [5]:
# Contar cuántas transacciones son legítimas (0) y cuántas son fraude (1)
print(df['isFraud'].value_counts())

# Lo mismo pero en porcentaje, para ver la proporción
print(df['isFraud'].value_counts(normalize=True) * 100)

isFraud
0    6354407
1       8213
Name: count, dtype: int64
isFraud
0    99.870918
1     0.129082
Name: proportion, dtype: float64


El dataset tiene 6,362,620 transacciones, de las cuales solo 8,213 (0.13%) son fraude. Los datos están muy desbalanceados, así que la exactitud (accuracy) por sí sola no sirve para evaluar el modelo; habrá que fijarse en precision y recall de la clase fraude.

In [6]:
# Eliminar columnas que no aportan para predecir el fraude
# - nameOrig y nameDest: son IDs de cuentas, casi únicos por fila
# - isFlaggedFraud: es una alerta de otro sistema, no la queremos usar
df = df.drop(columns=['nameOrig', 'nameDest', 'isFlaggedFraud'])

# Verificar las columnas que quedaron
print(df.columns)

Index(['step', 'type', 'amount', 'oldbalanceOrg', 'newbalanceOrig',
       'oldbalanceDest', 'newbalanceDest', 'isFraud'],
      dtype='object')


Eliminé nameOrig y nameDest porque son identificadores sin patrón útil, e isFlaggedFraud porque es la alerta de otro sistema. Quedan 7 variables para predecir y la variable objetivo isFraud.

In [7]:
# Convertir la columna de texto 'type' en columnas numéricas (one-hot encoding)
# drop_first=True quita una columna redundante (si todas son 0, se sabe cuál es)
df = pd.get_dummies(df, columns=['type'], drop_first=True, dtype=int)

# Ver cómo quedaron las primeras filas
df.head()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
0,1,9839.64,170136.0,160296.36,0.0,0.0,0,0,0,1,0
1,1,1864.28,21249.0,19384.72,0.0,0.0,0,0,0,1,0
2,1,181.00,181.0,0.00,0.0,0.0,1,0,0,0,1
3,1,181.00,181.0,0.00,21182.0,0.0,1,1,0,0,0
4,1,11668.14,41554.0,29885.86,0.0,0.0,0,0,0,1,0


Convertí la columna type en 4 columnas numéricas con one-hot encoding. Ahora todas las variables son numéricas y el modelo puede usarlas.

In [8]:
# X: todas las columnas excepto la que queremos predecir
X = df.drop(columns=['isFraud'])

# y: la columna objetivo (1 = fraude, 0 = legítima)
y = df['isFraud']

# Verificar tamaños: (filas, columnas)
print(X.shape)
print(y.shape)

(6362620, 10)
(6362620,)


Separé los datos en X (10 variables de entrada) e y (la variable isFraud). Ambos tienen 6,362,620 filas.

In [9]:
# Dividir: 80% para entrenar y 20% para probar
# stratify=y mantiene la misma proporción de fraudes en ambos grupos
# random_state=42 hace que la división salga igual cada vez que se ejecute
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Verificar tamaños y que la proporción de fraude se mantuvo
print(X_train.shape, X_test.shape)
print(y_train.mean() * 100, y_test.mean() * 100)

(5090096, 10) (1272524, 10)
0.12907418642005966 0.12911347840983745


Dividí los datos en 80% entrenamiento (5,090,096 filas) y 20% prueba (1,272,524 filas). Gracias a stratify, ambos conjuntos conservan la misma proporción de fraude (0.129%).

In [10]:
# Crear el escalador
scaler = StandardScaler()

# Aprender la escala con los datos de entrenamiento y aplicarla
X_train_scaled = scaler.fit_transform(X_train)

# Aplicar la MISMA escala a los datos de prueba (sin volver a aprender)
X_test_scaled = scaler.transform(X_test)

# Verificar: el promedio de cada columna en train debe ser ~0
print(X_train_scaled.mean(axis=0).round(2))

[-0.  0.  0. -0. -0. -0. -0. -0.  0.  0.]


Escalé las variables con StandardScaler para que todas tengan una escala similar (promedio 0). El escalador aprendió solo con los datos de entrenamiento y se aplicó igual a los de prueba, para no filtrar información del conjunto de prueba.

In [11]:
# Crear el modelo de regresión logística
# max_iter=1000 le da más intentos para encontrar la mejor solución
model = LogisticRegression(max_iter=1000)

# Entrenar el modelo con los datos de entrenamiento escalados
model.fit(X_train_scaled, y_train)

print("Modelo entrenado ✅")

Modelo entrenado ✅


Entrené un modelo de regresión logística con los datos de entrenamiento escalados. El modelo estima la probabilidad de que una transacción sea fraude.

In [12]:
# Predecir con los datos de prueba
y_pred = model.predict(X_test_scaled)

# Matriz de confusión: compara lo real contra lo predicho
print(confusion_matrix(y_test, y_pred))

# Reporte con precision, recall y f1-score por clase
print(classification_report(y_test, y_pred, digits=4))

[[1270817      64]
 [    994     649]]
              precision    recall  f1-score   support

           0     0.9992    0.9999    0.9996   1270881
           1     0.9102    0.3950    0.5509      1643

    accuracy                         0.9992   1272524
   macro avg     0.9547    0.6975    0.7753   1272524
weighted avg     0.9991    0.9992    0.9990   1272524



El modelo tiene una accuracy de 99.92%, pero no es una métrica confiable por el desbalance. Para la clase fraude, la precision es de 91% (casi no da falsas alarmas), pero el recall es de solo 39.5%: detectó 649 de 1,643 fraudes y se le escaparon 994. En un banco, dejar pasar fraudes es costoso, así que el modelo necesitaría mejorar su recall.

In [13]:
# Crear un segundo modelo que da más importancia a la clase fraude
# class_weight='balanced' compensa que haya muy pocos fraudes
model_bal = LogisticRegression(max_iter=1000, class_weight='balanced')

# Entrenar con los mismos datos escalados
model_bal.fit(X_train_scaled, y_train)

# Predecir con los datos de prueba
y_pred_bal = model_bal.predict(X_test_scaled)

# Evaluar igual que antes para poder comparar
print(confusion_matrix(y_test, y_pred_bal))
print(classification_report(y_test, y_pred_bal, digits=4))

[[1206228   64653]
 [     64    1579]]
              precision    recall  f1-score   support

           0     0.9999    0.9491    0.9739   1270881
           1     0.0238    0.9610    0.0465      1643

    accuracy                         0.9491   1272524
   macro avg     0.5119    0.9551    0.5102   1272524
weighted avg     0.9987    0.9491    0.9727   1272524



Con class_weight='balanced' el recall subió de 39.5% a 96.1% (solo se escaparon 64 fraudes), pero la precision bajó de 91% a 2.4% (64,653 falsas alarmas). Esto muestra el trade-off entre precision y recall. El modelo 1 conviene si las falsas alarmas son caras; el modelo 2, si dejar pasar un fraude es lo más grave. Un punto intermedio podría lograrse ajustando el umbral de decisión.